### Generator for new york taxi trip data

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,TimestampType
from pyspark.sql.functions import current_timestamp, year, month
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import random
taxi_schema=StructType([

    StructField("VendorID",IntegerType()),
    StructField("Passenger_count",IntegerType()),
    StructField("trip_pickup_datetime",TimestampType()),
    StructField("trip_dropoff_datetime",TimestampType()),
    StructField("trip_distance",DoubleType()),
    StructField("RateCodID",IntegerType()),
    StructField("Store_and_fwd_flag",StringType()),
    StructField("PULocationID",IntegerType()),
    StructField("DOLocationID",IntegerType()),
    StructField("Payment_type",IntegerType()),
    StructField("Fare_amount",DoubleType()),
    StructField("extra",DoubleType()),
    StructField("mta_tax",DoubleType()),
    StructField("tip_amount",DoubleType()),
    StructField("tolls_amount",DoubleType()),
    StructField("improvement_surcharge",DoubleType()),
    StructField("total_amount",DoubleType()),
    StructField("congestion_surcharge",DoubleType()),
    StructField("airport_fee",DoubleType())
])

In [0]:
# Read the file explicitly using the standardized local file system API identifier
raw_csv_path = "/Volumes/dev_nyc/landing/nyc_taxi_trip_duration/generated_trips_raw.csv"

bronze_df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .option("mode", "PERMISSIVE") # Prevents failures by handling bad lines safely
              .load(raw_csv_path))




In [0]:
# Verify schema layout works without execution errors
display(bronze_df.limit(10))

In [0]:
#3. Add explicit timestamps and extract temporal partition attributes
bronze_df = (bronze_df
    # Convert raw text strings into strong timestamp data types
    .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss"))
    # Generate partitioning columns from the pickup timestamp
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
)
display(bronze_df.limit(5))

In [0]:
#  Write to Bronze Table
# We partition by 'pickup_year' and 'pickup_month' to optimize downstream storage and querying performance.

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("pickup_year", "pickup_month") \
    .saveAsTable("dev_nyc.bronze.taxi_duartion")

print(f"Successfully wrote data to Bronze location: dev_nyc.bronze.taxi_duartion")
